# 03 · Silver accounts
Cleans the accounts lookup, enforces one row per account, and checks every transacting account can be found.

In [0]:
from pyspark.sql import functions as F

BRONZE = "workspace.aml_bronze"
SILVER = "workspace.aml_silver"

bronze_accounts = spark.table(f"{BRONZE}.accounts")
display(bronze_accounts.limit(5))

def normalize_bank(col):
    # Canonical bank ID: remove leading zeros, keep a single "0" if all zeros
    return F.regexp_replace(col, r"^0+(?=.)", "")

In [0]:
accounts = (
    bronze_accounts
    .select(
        F.concat_ws("_", normalize_bank(F.trim("bank_id")), F.trim("account_number")).alias("account_id"),
        normalize_bank(F.trim("bank_id")).alias("bank_id"),
        F.trim("bank_name").alias("bank_name"),
        F.trim("account_number").alias("account_number"),
        F.trim("entity_id").alias("entity_id"),
        F.trim("entity_name").alias("entity_name"),
        "_source_file",
        F.current_timestamp().alias("_processed_at"),
    )
)

In [0]:
total_rows = accounts.count()
unique_accounts = accounts.select("account_id").distinct().count()

print(f"Rows:            {total_rows:,}")
print(f"Unique accounts: {unique_accounts:,}")

In [0]:
txns = spark.table(f"{SILVER}.transactions")

transacting = (
    txns.select(F.col("from_account_id").alias("account_id"))
    .union(txns.select(F.col("to_account_id").alias("account_id")))
    .distinct()
)

missing = transacting.join(accounts, on="account_id", how="left_anti")

print(f"Transacting accounts:  {transacting.count():,}")
display(missing.limit(10))

In [0]:
missing_count = missing.count()

print(f"Rows:                  {total_rows:,}")
print(f"Unique accounts:       {unique_accounts:,}")
print(f"Missing from accounts: {missing_count:,}")

assert unique_accounts == total_rows, "account_id is not unique"
assert missing_count == 0, "Some transacting accounts are missing from the accounts table"

print("Quality gate passed")

(
    accounts.write
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{SILVER}.accounts")
)